## Vẽ hình 2 màu

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# Đọc dữ liệu
opl_df = pd.read_excel("/content/inka_save_H2228_1_opl.xlsx")
ln_df = pd.read_excel("/content/inka_save_H2228_1_log2_new_method.xlsx")

# Các kinase cần làm nổi bật
highlight_kinases = {"ALK", "PTK2", "SRC", "LYN", "EGFR"}

# Các chỉ số sẽ được vẽ
metrics = ['Kinome', 'ActLoop', 'PSP', 'NWK', 'Score']

# Tạo PDF
with PdfPages("inka_comparison_5x2.pdf") as pdf:
    # Tạo 5 dòng, mỗi dòng 2 cột: OPL và ln
    fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(12, 20))

    for i, metric in enumerate(metrics):
        # Dữ liệu OPL
        df_opl = opl_df.sort_values(by=metric, ascending=False).head(20)[::-1]
        colors_opl = ['purple' if k in highlight_kinases else 'white' for k in df_opl['Kinase']]
        axes[i, 0].barh(df_opl['Kinase'], df_opl[metric], color=colors_opl, edgecolor='black')
        axes[i, 0].set_title(f"OPL - {metric}", fontsize=12, weight='bold')
        axes[i, 0].set_facecolor('#e0f5c8')
        axes[i, 0].tick_params(axis='y', labelsize=8)

        # Dữ liệu ln
        df_ln = ln_df.sort_values(by=metric, ascending=False).head(20)[::-1]
        colors_ln = ['purple' if k in highlight_kinases else 'white' for k in df_ln['Kinase']]
        axes[i, 1].barh(df_ln['Kinase'], df_ln[metric], color=colors_ln, edgecolor='black')
        axes[i, 1].set_title(f"Log 10 Norm Intensity - {metric}", fontsize=12, weight='bold')
        axes[i, 1].set_facecolor('#e0f5c8')
        axes[i, 1].tick_params(axis='y', labelsize=8)

    # Căn chỉnh bố cục
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close()


## Vẽ hình nhiều màu

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import colorsys

opl_df  = pd.read_excel("/content/inka_save_H2228_1_opl.xlsx")
log2_df = pd.read_excel("/content/inka_save_H2228_1_log2.xlsx")

metrics  = ['Kinome', 'ActLoop', 'PSP', 'NWK', 'Score']
THRESH   = 5   # ngưỡng lệch rank

# ---- 1. Danh sách màu tương phản cao (Set1 + Dark2) ----
base_colors = [
    "#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00",
    "#ffff33", "#a65628", "#f781bf", "#999999", "#66c2a5",
    "#fc8d62", "#8da0cb", "#e78ac3", "#a6d854", "#ffd92f",
    "#e5c494", "#b3b3b3"
]

def generate_extra_colors(n):
    """Sinh thêm màu HSV cách đều khi > len(base_colors)."""
    return [colorsys.hsv_to_rgb(i/n, 0.75, 0.95) for i in range(n)]

with PdfPages("inka_comparison_H2228_1_opl_log2_rankdiff>5_highcontrast.pdf") as pdf:
    fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(12, 20))

    for i, metric in enumerate(metrics):
        # ---- 2. Tính rank & highlight set ----
        diff = (opl_df[metric].rank(method="min", ascending=False) - log2_df[metric].rank(method="min", ascending=False)).abs()
        highlight_set = opl_df['Kinase'][diff > THRESH].tolist()

        print(f"\n--- Metric: {metric} ---")

        rank_opl  = opl_df[metric].rank(method="min", ascending=False)
        rank_log2 = log2_df[metric].rank(method="min", ascending=False)
        rank_diff = (rank_opl - rank_log2).abs()

        # Lấy top 20 theo mỗi bảng
        top20_opl  = set(opl_df.sort_values(metric, ascending=False).head(20)['Kinase'])
        top20_log2 = set(log2_df.sort_values(metric, ascending=False).head(20)['Kinase'])

        # Hợp của 2 top20
        top_union = top20_opl.union(top20_log2)

        # Lọc những kinase trong top_union có chênh lệch rank > THRESH
        for idx in opl_df.index:
            kinase = opl_df.loc[idx, 'Kinase']
            if kinase in top_union and rank_diff.loc[idx] > THRESH:
                ro = int(rank_opl.loc[idx])
                rl = int(rank_log2.loc[idx])
                d  = int(rank_diff.loc[idx])
                print(f"{kinase:25s} | Rank OPL: {ro:3d} | Rank Log2: {rl:3d} | ΔRank: {d}")



        # Nếu cần nhiều màu hơn base_colors
        if len(highlight_set) > len(base_colors):
            extra = generate_extra_colors(len(highlight_set) - len(base_colors))
            highlight_colors = base_colors + [plt.matplotlib.colors.to_hex(c) for c in extra]
        else:
            highlight_colors = base_colors

        color_map = {k: highlight_colors[j % len(highlight_colors)]
                     for j, k in enumerate(highlight_set)}

        # ---- 3. Vẽ OPL ----
        df_opl = opl_df.sort_values(metric, ascending=False).head(20)[::-1]
        colors_opl = [color_map.get(k, 'white') for k in df_opl['Kinase']]
        axes[i, 0].barh(df_opl['Kinase'], df_opl[metric], color=colors_opl, edgecolor='black')
        axes[i, 0].set_title(f"OPL – {metric}", fontsize=11, weight='bold')
        axes[i, 0].set_facecolor('#e0f5c8')
        axes[i, 0].tick_params(axis='y', labelsize=7)

        # ---- 4. Vẽ Log2 ----
        df_log2 = log2_df.sort_values(metric, ascending=False).head(20)[::-1]
        colors_log2 = [color_map.get(k, 'white') for k in df_log2['Kinase']]
        axes[i, 1].barh(df_log2['Kinase'], df_log2[metric], color=colors_log2, edgecolor='black')
        axes[i, 1].set_title(f"Log2 – {metric}", fontsize=11, weight='bold')
        axes[i, 1].set_facecolor('#e0f5c8')
        axes[i, 1].tick_params(axis='y', labelsize=7)

    plt.tight_layout()
    pdf.savefig(fig)
    plt.close()
